# 12 · Statistical comparison

Pre-specified paired comparisons on the same questions, 95% cluster-bootstrap intervals over scenarios, exact McNemar tests for Recall@1 and Holm correction across the primary held-out comparisons. An improvement is reported as supported only when its interval excludes zero. Writes the Milestone 5 report and abstract.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
import glob
def load_exp(ds):
    f = A.RESULTS / f"experiments_{ds}.csv"
    if not f.exists():
        return None
    df = A.load_retrieval(f.name)
    df["latency_ms"] = df["total_ms"]
    return df
EXP = {ds: load_exp(ds) for ds in ("pilot_v0.1", "heldout_v0.2")}
for ds, df in EXP.items():
    print(ds, "not evaluated (no experiments file)" if df is None else f"{len(df)} rows, methods: {sorted(df['method'].unique())}")
MANIFESTS = {Path(f).stem.replace('reranked_', '').replace('.manifest', ''): json.loads(Path(f).read_text())
             for f in glob.glob(str(A.RESULTS / 'reranked_*.manifest.json'))}

In [ ]:
PAIRS = [("E2", "B0", "improved hybrid vs dense"), ("E2", "E1", "improved vs original hybrid"),
         ("B0FR_bge-m3", "B0", "B0R: dense pool + bge-m3 vs dense"), ("B0FR_bge-m3", "B0F", "dense (no threshold) + bge-m3"),
         ("E3_bge-m3", "E2", "hybrid + bge-m3 vs hybrid"), ("E3_mminilm", "E2", "hybrid + mMiniLM vs hybrid"),
         ("E4_bge-m3", "E3_bge-m3", "+ evidence selection vs reranked hybrid"), ("E3_bge-m3", "B0", "reranked hybrid vs dense")]
rows = []
for ds, df in EXP.items():
    if df is None: continue
    for a, b, what in PAIRS:
        if {a, b} <= set(df.method):
            for col in ("r1", "r3", "r5", "rr", "ndcg5", "ndcg10"):
                r = A.paired_difference(df, a, b, col); r.update(dataset=ds, what=what); rows.append(r)
cmp = pd.DataFrame(rows)
primary = cmp[(cmp.dataset == "heldout_v0.2") & (cmp.metric == "rr")].copy()
# Holm correction over the primary (held-out MRR) comparisons, using the bootstrap two-sided p-value approximation
def boot_p(r):
    se = (r["hi"] - r["lo"]) / (2 * 1.96)
    return float(2 * (1 - scipy.stats.norm.cdf(abs(r["diff"]) / se))) if se > 0 else 1.0
import scipy.stats
primary["p_approx"] = primary.apply(boot_p, axis=1)
primary = primary.sort_values("p_approx")
m = len(primary); primary["holm_threshold"] = [0.05 / (m - i) for i in range(m)]
reject, still = [], True
for p_, t_ in zip(primary["p_approx"], primary["holm_threshold"]):
    still = still and p_ <= t_; reject.append(still)
primary["supported_after_holm"] = reject
cmp.to_csv(A.RESULTS / "m5_paired_comparisons.csv", index=False)
primary[["a", "b", "what", "n", "mean_a", "mean_b", "diff", "lo", "hi", "p_approx", "holm_threshold", "supported_after_holm"]].round(4)

In [ ]:
h = cmp[(cmp.dataset == "heldout_v0.2") & (cmp.metric.isin(["r1", "rr"]))]
labels = [f"{a} vs {b}" for a, b, _ in PAIRS if not h[(h.a == a) & (h.b == b)].empty]
fig, ax = plt.subplots(figsize=(10, 0.55 * len(labels) + 1.6))
for j, (metric, color) in enumerate((("r1", A.COLORS["B0"]), ("rr", A.COLORS["E1"]))):
    sub = h[h.metric == metric].assign(label=lambda d: d.a + " vs " + d.b).set_index("label").reindex(labels)
    y = np.arange(len(labels)) + (j - 0.5) * 0.3
    ax.errorbar(100 * sub["diff"], y, xerr=[100 * (sub["diff"] - sub["lo"]), 100 * (sub["hi"] - sub["diff"])], fmt="o",
                color=color, capsize=3, markersize=7, linewidth=1.6, label={"r1": "Recall@1", "rr": "MRR"}[metric])
    for xv, yv in zip(100 * sub["diff"], y):
        ax.text(xv, yv + 0.13, f"{xv:+.1f}", ha="center", fontsize=8, color=A.INK)
ax.axvline(0, color=A.INK2, linewidth=1); ax.set_yticks(np.arange(len(labels)), labels); ax.invert_yaxis()
ax.set_ylim(len(labels) - 0.4, -0.75); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xlabel("paired difference, percentage points (95% cluster-bootstrap CI)")
ax.set_title("Held-out v0.2 (n = 100 questions, 20 scenarios) - pre-specified comparisons")
ax.legend(loc="lower right"); fig.tight_layout(); print(A.save(fig, "12_forest_plot"))

## Subgroups (held-out): language, cross-language, document length, question type

In [ ]:
df = EXP["heldout_v0.2"]
manifest = {d["id"]: d for d in json.loads((A.SNAP / "corpus_manifest.json").read_text())["items"]}
bench = {r["question_id"]: r for r in json.loads((A.DATA / "benchmarks/retrieval_heldout_v0.2.json").read_text())["records"]}
chunks = pd.read_csv(A.SNAP / "chunk_metrics.csv"); per_doc = chunks[chunks.active].groupby("document_id").size()
aliases = json.loads((A.DATA / "id_aliases.json").read_text())["aliases"]
def n_chunks(pid): return per_doc.get(aliases.get(pid, pid), np.nan)
df["cross_language"] = df.question_id.map(lambda q: bench[q]["language"] != manifest[bench[q]["primary_document_id"]]["language"])
df["long_document"] = df.question_id.map(lambda q: n_chunks(bench[q]["primary_document_id"]) >= 30)
sel = ["B0", "E2", "E3_bge-m3", "E4_bge-m3"]
sub = df[df.method.isin(sel)]
for key in ("language", "cross_language", "long_document", "difficulty", "category"):
    t = A.summarize(sub, ["method", key], metrics=("r1", "rr")).pivot(index=key, columns="method", values="rr")
    display(Markdown(f"**MRR by {key}** (n per cell = questions in that group)")); display(t.round(3))

In [ ]:
res = A.summarize(df, ["method"], metrics=("r1", "r3", "r5", "r10", "rr", "ndcg5", "ndcg10")).set_index("method")
dev = A.summarize(EXP["pilot_v0.1"], ["method"], metrics=("r1", "rr")).set_index("method")
stats = json.loads((A.SNAP / "dataset_statistics.json").read_text())
def ci(m, k="rr"):
    if m not in res.index:
        return "not evaluated"
    return f"{res.loc[m, k]:.3f} [{res.loc[m, k+'_lo']:.3f}, {res.loc[m, k+'_hi']:.3f}]"
def pp(a, b, metric="rr"):
    m = cmp[(cmp.dataset == "heldout_v0.2") & (cmp.a == a) & (cmp.b == b) & (cmp.metric == metric)]
    if m.empty:
        return "not evaluated"
    r = m.iloc[0]
    return f"{100*r['diff']:+.1f} pp [{100*r['lo']:+.1f}, {100*r['hi']:+.1f}]" + (" (supported)" if r["ci_excludes_zero"] else " (inconclusive)")
table = pd.DataFrame([dict(method=m, **{k: f"{res.loc[m, k]:.3f}" for k in ("r1", "r3", "r5", "r10", "ndcg5", "ndcg10")},
                           MRR=ci(m)) for m in res.index])
table.to_csv(A.REPORTS / "m5_ablation_table.csv", index=False)
lat = df[df.reranker.isna() | (df.get("rerank_device", "mps") != "cpu")].groupby("method")["total_ms"].median()   # GPU reranking rows
report = f"""# Milestone 5 - multilingual reranking and error analysis (preliminary)

All results below are **preliminary**: questions and relevance labels are AI-drafted and not human-reviewed. The held-out
set v0.2 was never used for tuning but its results were inspected in Milestone 4 ("inspected, untuned"). Local
experiments (Apple M1; reranking on the GPU via MPS, CPU for the first development rows) on identical frozen candidate pools (≤ 20 chunks); corpus snapshot
{json.loads((A.SNAP / 'corpus_manifest.json').read_text())['snapshot_sha256_16']} ({A.SNAPSHOT}); embedding multilingual-e5-small (ONNX int8).

## Held-out results (n = 100 questions, 20 scenarios, 5 languages)
{table.to_markdown(index=False)}

`[a, b]` = 95% cluster-bootstrap interval over scenarios.

## Key paired comparisons (held-out MRR)
- E3 (hybrid + bge-reranker-v2-m3) vs E2: {pp('E3_bge-m3', 'E2')}
- E3 (hybrid + mMiniLM) vs E2: {pp('E3_mminilm', 'E2')}
- B0R (dense top-20 pool without threshold + bge-m3, method B0FR) vs B0: {pp('B0FR_bge-m3', 'B0')}
- E4 (+ evidence selection) vs E3 (bge-m3): {pp('E4_bge-m3', 'E3_bge-m3')}
- E2 vs B0: {pp('E2', 'B0')}; E2 vs E1: {pp('E2', 'E1')}

Holm-corrected held-out MRR comparisons supported: {', '.join(f"{r.a} vs {r.b}" for r in primary.itertuples() if r.supported_after_holm) or 'none'}.

## Cost
Median local time per question (Apple M1; first stage CPU, reranking GPU/MPS): E2 {lat.get('E2', float('nan')):.0f} ms; E3 bge-m3 {lat.get('E3_bge-m3', float('nan')):.0f} ms;
E3 mMiniLM {lat.get('E3_mminilm', float('nan')):.0f} ms.

## Not evaluated
Answer quality per retrieval pipeline (no working primary Gemini model locally, no human scores); human-reviewed final test set.
"""
(A.REPORTS / "milestone5_results.md").write_text(report)
display(Markdown(report))

In [ ]:
abstract = f"""**Draft abstract (Milestone 5, preliminary - AI-drafted labels, not human-reviewed).**

We study retrieval for a multilingual assistant that answers international students' questions in Korean, English,
Indonesian, Vietnamese and Chinese from {stats['documents']['active_searchable']} approved university and government
documents written mainly in English and Korean ({stats['chunks']['active']} chunks; multilingual-e5-small in
PostgreSQL/pgvector). An error analysis of 200 synthetic questions shows that most first-rank failures arise when a
document in the question's own script outranks the relevant document in another language, followed by keyword
ambiguity and dominance of one long multi-topic guide. On a held-out set (100 questions, 20 scenarios) we compare dense
retrieval, two hybrid retrievers and off-the-shelf multilingual cross-encoder reranking on identical candidate pools:
the improved hybrid reaches MRR {ci('E2')}, and reranking it with bge-reranker-v2-m3 gives {ci('E3_bge-m3')}
(paired difference {pp('E3_bge-m3', 'E2')}) at a median local cost (Apple M1, GPU reranking) of {lat.get('E3_bge-m3', float('nan')):.0f} ms per question.
Answer-level effects were not evaluated. All evaluation data, scripts and notebooks are released for review."""
(A.REPORTS / "abstract_draft_m5.md").write_text(abstract + "\n"); display(Markdown(abstract))